# BS5 D-R-reset: Kaggle
GPU + Internet. Source Dataset is already extracted. Two main runs only: 551/552 ? 24k. See BS5_DR_RESET_KAGGLE.md for restore instructions.


In [ ]:
from pathlib import Path
import os
import sys
import json
import subprocess

os.environ["PYTHONDONTWRITEBYTECODE"] = "1"
sys.dont_write_bytecode = True
SOURCE_FILTER = ""  # Điền tên Dataset nếu có nhiều bản source.
candidates = sorted({
    p.parents[2] for p in Path("/kaggle/input").rglob("run_bs5_direct_replay_reset.py")
    if p.parent.name == "training" and p.parents[1].name == "scripts"
    and SOURCE_FILTER in str(p)
    and (p.parents[2] / "configs/bs5_direct_replay_reset_baseline_v2.json").is_file()
    and (p.parents[2] / "AGENTS.md").is_file()
})
if len(candidates) != 1:
    raise RuntimeError(f"Cần đúng một source; tìm thấy {candidates}. Chỉnh SOURCE_FILTER.")
ROOT = candidates[0].resolve()
print("Source:", ROOT)


In [ ]:
import importlib.metadata

os.chdir(ROOT)
# Cài trước khi import torch trong kernel. Runner luôn chạy bằng subprocess mới.
if importlib.metadata.version("torch").split("+")[0] != "2.11.0":
    subprocess.run([
        sys.executable, "-m", "pip", "install", "--no-cache-dir",
        "torch==2.11.0", "--index-url", "https://download.pytorch.org/whl/cu128"
    ], check=True)
if importlib.metadata.version("numpy") != "2.1.3":
    subprocess.run([sys.executable, "-m", "pip", "install", "--no-cache-dir", "numpy==2.1.3"], check=True)
subprocess.run([sys.executable, "-B", "-c",
    "import torch,numpy; print('torch',torch.__version__,'numpy',numpy.__version__); "
    "assert torch.cuda.is_available(), 'Bật GPU Accelerator'; print(torch.cuda.get_device_name(0))"
], check=True)
WORK = Path("/kaggle/working")
RUN_NAME = "bs5_dr_reset_20261023"  # Đổi tên khi mở thí nghiệm độc lập.
if not RUN_NAME or RUN_NAME in {".", ".."} or any(c in RUN_NAME for c in "/\\"):
    raise ValueError("RUN_NAME phải là một tên thư mục")
OUT = WORK / RUN_NAME
SMOKE = WORK / (RUN_NAME + "_smoke")
BASE = [sys.executable, "-B", "-u", "-m", "scripts.training.run_bs5_direct_replay_reset"]
def run_reset(*args):
    subprocess.run(BASE + list(args), cwd=ROOT, check=True, env=os.environ.copy())
print("CWD:", Path.cwd(), "\nMain output:", OUT)


In [ ]:
if not (SMOKE / "protocol.json").is_file():
    run_reset("--prepare", "--smoke", "--device", "cuda", "--outdir", str(SMOKE))
smoke_registry = json.loads((SMOKE / "run_registry.json").read_text())
if smoke_registry["status"] != "complete":
    run_reset("--run", "--resume", "--export-policy", "final", "--outdir", str(SMOKE))
elif not (SMOKE / "analysis/verification.json").is_file():
    run_reset("--export", "--outdir", str(SMOKE))
assert json.loads((SMOKE / "analysis/verification.json").read_text())["complete"]
timings = [json.loads(p.read_text()) for p in (SMOKE / "timing").glob("*.json")]
outer = {k: sum(t["outer"].get(k, 0) for t in timings)
         for k in ("setup", "train_loop", "eval", "persistence", "analysis", "export", "other")}
# Chỉ là ước lượng: smoke horizon6, main150; lifecycle/oracle slot76 và kích thước
# replay/evidence lớn hơn khiến phép ngoại suy có sai số đáng kể.
estimate_h = (outer["train_loop"] / 768 * 48000 + outer["eval"] / 216 * 27000) * 1.5 / 3600
print("Smoke outer seconds:", outer)
print("Ước lượng train+eval ×1.5 (giờ), chưa gồm export lớn:", round(estimate_h, 2))
if not (OUT / "protocol.json").is_file():
    run_reset("--prepare", "--outdir", str(OUT))
# Prepare kiểm initial weights từng seed so C-R/D-R cũ; sai hash thì dừng trước rollout.
run_reset("--run", "--resume", "--outdir", str(OUT))


In [ ]:
from IPython.display import FileLink, display
import hashlib

def canonical_hash(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, separators=(",", ":"), allow_nan=False).encode()).hexdigest()
# Chỉ chạy sau khi cell 3 đã dừng. Reconcile không train, không clear replay mới.
run_reset("--reconcile", "--outdir", str(OUT))
protocol = json.loads((OUT / "protocol.json").read_text())
registry = json.loads((OUT / "run_registry.json").read_text())
commits = {name: json.loads((OUT / "runs" / name / "commit.json").read_text())
           for name, entry in registry["runs"].items() if entry["status"] != "not_started"}
snapshot = canonical_hash(dict(protocol=protocol, commits=commits))
latest_path = OUT / "exports/latest.json"
latest = json.loads(latest_path.read_text()) if latest_path.is_file() else {}
if latest.get("snapshot_sha256") != snapshot:
    run_reset("--export", "--outdir", str(OUT))
    latest = json.loads(latest_path.read_text())
print("Status:", registry["status"], "| Tải review ZIP trước để xem kết quả.")
for name, expected in sorted(latest["bundles"].items(), key=lambda item: "_review.zip" not in item[0]):
    path = OUT / "exports" / name
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    assert digest.hexdigest() == expected, f"ZIP hash mismatch: {name}"
    print(name, f"{path.stat().st_size / 1024**2:.2f} MiB")
    display(FileLink(str(path)))
display(FileLink(str(latest_path)))  # Receipt có timer gồm export/review hoàn tất.
